# Анализ разнообразия ошибок (Error Diversity Analysis)

**Цель:** Найти эксперименты, которые верно классифицируют *разные* тестовые примеры —
такие модели взаимно дополняют друг друга и дают наибольший выигрыш при объединении в ансамбль.

**Метрики разнообразия:**
- **Q-статистика** (Yule, 1900): 0 = независимые ошибки, 1 = идентичные, -1 = полностью комплементарные
- **Корреляция ошибок ρ**: аналог Q, но чувствительнее к дисбалансу классов
- **Процент «спасаемых» ошибок**: доля примеров, где хотя бы одна из двух моделей права

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

sys.path.insert(0, str(Path("../../").resolve()))
from shared import config

EXPERIMENTS_DIR = Path("../../")

In [ ]:
# Ищем все test_predictions*.csv во всех экспериментах
pred_files = sorted(EXPERIMENTS_DIR.glob("*/*/test_predictions*.csv"))

dfs = {}
for f in pred_files:
    # Имя модели: папка эксперимента (+ суффикс если несколько файлов)
    exp_name = f.parent.name
    suffix = f.stem.replace("test_predictions", "").lstrip("_")
    key = suffix if suffix else exp_name
    try:
        df = pd.read_csv(f)
        dfs[key] = df.set_index("path")
        print(f"  ✓ {key:40s}  n={len(df)}")
    except Exception as e:
        print(f"  ✗ {key}: {e}")

print(f"\nЗагружено моделей: {len(dfs)}")

In [ ]:
# Общие пути (пересечение всех тест-сетов — должны совпадать)
common_paths = None
for df in dfs.values():
    paths = set(df.index)
    common_paths = paths if common_paths is None else common_paths & paths

common_paths = sorted(common_paths)
print(f"Общих тестовых примеров: {len(common_paths)}")

# Матрица: строки = примеры, столбцы = модели, значение = 1 (верно) / 0 (ошибка)
correct_matrix = pd.DataFrame(index=common_paths)
proba_matrix   = pd.DataFrame(index=common_paths)

for name, df in dfs.items():
    df_aligned = df.loc[common_paths]
    correct_matrix[name] = (df_aligned["y_pred"] == df_aligned["y_true"]).astype(int)
    proba_matrix[name]   = df_aligned["y_proba"]

y_true = dfs[list(dfs.keys())[0]].loc[common_paths, "y_true"].values
print(f"Класс BAD в тесте: {y_true.sum()} / {len(y_true)} ({y_true.mean()*100:.1f}%)")
print(f"\nТочность по моделям:")
print(correct_matrix.mean().sort_values(ascending=False).to_string())

In [ ]:
def q_statistic(c_i, c_j):
    """Q-статистика Юла между двумя бинарными векторами правильных классификаций."""
    N11 = ((c_i == 1) & (c_j == 1)).sum()
    N00 = ((c_i == 0) & (c_j == 0)).sum()
    N10 = ((c_i == 1) & (c_j == 0)).sum()
    N01 = ((c_i == 0) & (c_j == 1)).sum()
    denom = N11 * N00 + N01 * N10
    return (N11 * N00 - N01 * N10) / denom if denom > 0 else 0.0

def error_correlation(c_i, c_j):
    """Коэффициент корреляции ошибок (формула Kuncheva)."""
    N = len(c_i)
    N11 = ((c_i == 1) & (c_j == 1)).sum()
    N00 = ((c_i == 0) & (c_j == 0)).sum()
    N10 = ((c_i == 1) & (c_j == 0)).sum()
    N01 = ((c_i == 0) & (c_j == 1)).sum()
    denom = np.sqrt((N11+N10)*(N01+N00)*(N11+N01)*(N10+N00))
    return (N11*N00 - N01*N10) / denom if denom > 0 else 0.0

def saveable_errors(c_i, c_j):
    """Доля примеров, где хотя бы одна из двух моделей права."""
    return ((c_i == 1) | (c_j == 1)).mean()

models = list(correct_matrix.columns)
n = len(models)

Q   = pd.DataFrame(np.eye(n), index=models, columns=models)
RHO = pd.DataFrame(np.eye(n), index=models, columns=models)
SAV = pd.DataFrame(np.ones((n, n)), index=models, columns=models)

rows = []
for m1, m2 in combinations(models, 2):
    c1 = correct_matrix[m1].values
    c2 = correct_matrix[m2].values
    q  = q_statistic(c1, c2)
    rho = error_correlation(c1, c2)
    sav = saveable_errors(c1, c2)
    Q.loc[m1, m2] = Q.loc[m2, m1] = q
    RHO.loc[m1, m2] = RHO.loc[m2, m1] = rho
    SAV.loc[m1, m2] = SAV.loc[m2, m1] = sav
    rows.append({"model_1": m1, "model_2": m2,
                 "Q": round(q, 3), "rho": round(rho, 3),
                 "saveable": round(sav, 3),
                 "acc_1": round(correct_matrix[m1].mean(), 3),
                 "acc_2": round(correct_matrix[m2].mean(), 3)})

df_pairs = pd.DataFrame(rows).sort_values("Q")
print("Топ-10 наиболее РАЗНООБРАЗНЫХ пар (низкий Q):")
display(df_pairs.head(10))
print("\nТоп-10 наиболее ПОХОЖИХ пар (высокий Q):")
display(df_pairs.tail(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Q-statistic heatmap
sns.heatmap(Q, annot=True, fmt=".2f", cmap="RdYlGn_r",
            vmin=-1, vmax=1, center=0,
            ax=axes[0], linewidths=0.5)
axes[0].set_title("Q-статистика (ниже = разнообразнее)", fontsize=13)
axes[0].tick_params(axis='x', rotation=45)

# Saveable errors heatmap
sns.heatmap(SAV, annot=True, fmt=".2f", cmap="YlGn",
            vmin=0, vmax=1,
            ax=axes[1], linewidths=0.5)
axes[1].set_title("Доля примеров, где хотя бы 1 модель права", fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("diversity_heatmaps.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# «Трудные» примеры — все модели ошибаются
n_correct = correct_matrix.sum(axis=1)  # сколько моделей верно классифицировало

print(f"Все модели правы:   {(n_correct == n).sum():4d} примеров ({(n_correct == n).mean()*100:.1f}%)")
print(f"Все модели ошиблись: {(n_correct == 0).sum():4d} примеров ({(n_correct == 0).mean()*100:.1f}%)")
print(f"Хотя бы 1 права:    {(n_correct >= 1).sum():4d} примеров ({(n_correct >= 1).mean()*100:.1f}%)")

# Разбивка по классу
hard_bad  = (n_correct == 0) & (y_true == 1)
hard_good = (n_correct == 0) & (y_true == 0)
print(f"\nТрудные BAD  (дефект, все ошиблись): {hard_bad.sum()}")
print(f"Трудные GOOD (норма, все ошиблись):  {hard_good.sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
bins = range(n + 2)
ax.hist(n_correct[y_true == 1], bins=bins, alpha=0.7, label="BAD (дефект)", align="left")
ax.hist(n_correct[y_true == 0], bins=bins, alpha=0.7, label="GOOD (норма)", align="left")
ax.set_xlabel("Количество моделей, верно классифицировавших пример")
ax.set_ylabel("Количество примеров")
ax.set_title("Распределение «сложности» тестовых примеров")
ax.set_xticks(range(n + 1))
ax.legend()
plt.tight_layout()
plt.savefig("hard_samples_distribution.png", dpi=120)
plt.show()

In [ ]:
def oracle_accuracy(models_subset, correct_mat, majority=False):
    """
    majority=False: «оракул» — правильно если хотя бы 1 модель права.
    majority=True:  большинство голосов (мажоритарное голосование).
    """
    sub = correct_mat[list(models_subset)]
    if majority:
        return (sub.sum(axis=1) > len(models_subset) / 2).mean()
    else:
        return (sub.sum(axis=1) >= 1).mean()

print("Лучшие ТРОЙКИ моделей по оракульной точности (хотя бы 1 права):")
triple_rows = []
for trio in combinations(models, 3):
    oracle_acc = oracle_accuracy(trio, correct_matrix, majority=False)
    maj_acc    = oracle_accuracy(trio, correct_matrix, majority=True)
    avg_q      = np.mean([Q.loc[a, b] for a, b in combinations(trio, 2)])
    triple_rows.append({
        "models": " + ".join(trio),
        "oracle_acc": round(oracle_acc, 3),
        "majority_acc": round(maj_acc, 3),
        "avg_Q": round(avg_q, 3),
    })

df_triples = (pd.DataFrame(triple_rows)
              .sort_values("oracle_acc", ascending=False))
print("По максимальному потенциалу (оракул):")
display(df_triples.head(10))

print("\nПо точности большинства голосов:")
display(df_triples.sort_values("majority_acc", ascending=False).head(10))

In [ ]:
# Уникальные ошибки и уникальные победы каждой модели
print(f"{'Модель':40s}  {'Acc':>6}  {'Уник.правильных':>16}  {'Уник.ошибок':>12}")
print("-" * 80)
for m in models:
    acc = correct_matrix[m].mean()
    # Примеры, которые только эта модель классифицирует правильно
    others_correct = correct_matrix.drop(columns=m).sum(axis=1)
    unique_correct = ((correct_matrix[m] == 1) & (others_correct == 0)).sum()
    # Примеры, в которых только эта модель ошибается
    others_correct2 = correct_matrix.drop(columns=m).sum(axis=1)
    unique_wrong = ((correct_matrix[m] == 0) & (others_correct2 == len(models) - 1)).sum()
    print(f"{m:40s}  {acc:>6.3f}  {unique_correct:>16}  {unique_wrong:>12}")